# Aula 1 — Transformer simplificado em PyTorch

**Duração sugerida:** ~55 min (teoria no quadro/slides + este notebook)
**Pré-requisitos:** Python, PyTorch básico (tensores, `nn.Module`, `nn.Linear`).
**Execução:** somente CPU; nesta aula rodamos **apenas inferência** (o treino foi feito antes com `train_mini_transformer.py`).

### Roteiro do notebook

| # | Seção | O que mostra | Tempo |
|---|-------|--------------|-------|
| 1 | Das RNNs ao Transformer | por que a recorrência limita (gradiente e paralelismo) | 6 min |
| 2 | Embeddings e codificação posicional | como o modelo "sabe" a ordem | 8 min |
| 3 | Atenção por produto escalar | $\text{softmax}(QK^T/\sqrt{d_k})V$, com conta conferível à mão | 10 min |
| 4 | Atenção multi-cabeça | `view` / `transpose` e verificação contra o PyTorch | 8 min |
| 5 | Blocos do encoder e do decoder | residual, LayerNorm, FFN, máscara causal, cross-attention | 8 min |
| 6 | Transformer completo e rastreio de *shapes* | cada tensor, de ponta a ponta | 5 min |
| 7 | Carregar pesos e fazer inferência | decodificação gulosa na tarefa "inverter sequência" | 5 min |
| 8 | Mapas de atenção | o que cada atenção aprendeu | 5 min |

Referências: Vaswani et al. (2017), *Attention Is All You Need* (seções 3.1–3.5); Géron, cap. 16; Warfield (*Transformers, Positional Encoding, Multi-Headed Attention — Intuitively and Exhaustively Explained*).

In [ ]:
import os

# Evita a queda do kernel no Windows/conda (duas cópias da biblioteca OpenMP: PyTorch + numpy/MKL)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import math
import os
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
torch.set_printoptions(precision=4, sci_mode=False, linewidth=120)
print("PyTorch", torch.__version__)

## 1. Das RNNs ao Transformer

Em uma RNN, o estado oculto é atualizado **um token por vez**:

$$h_t = \tanh(W_{xh}\,x_t + W_{hh}\,h_{t-1} + b)$$

Isso traz dois problemas que o Transformer resolve:

1. **Sequencial:** para calcular $h_t$ é preciso ter calculado $h_{t-1}$, então não há paralelismo ao longo da sequência.
2. **Caminho longo:** a informação do token 1 chega ao token 50 depois de passar por 49 multiplicações por $W_{hh}$. O gradiente tende a **sumir** (ou explodir) nesse caminho.

Na autoatenção, qualquer par de posições se conecta **diretamente** (caminho de comprimento 1). Vamos medir isso: o quanto a saída da última posição "sente" cada token de entrada, via norma do gradiente.

In [ ]:
torch.manual_seed(0)
T, d = 50, 16
x = torch.randn(1, T, d, requires_grad=True)

# --- RNN simples (tanh) ---
rnn = nn.RNN(d, d, nonlinearity="tanh", batch_first=True)
out_rnn, _ = rnn(x)
out_rnn[0, -1].sum().backward()
g_rnn = x.grad[0].norm(dim=-1).clone()
x.grad = None

# --- Autoatenção (sem parâmetros aprendidos, só para ilustrar o caminho curto) ---
def self_attention_demo(x):
    scores = x @ x.transpose(-2, -1) / math.sqrt(x.size(-1))
    return F.softmax(scores, dim=-1) @ x

out_att = self_attention_demo(x)
out_att[0, -1].sum().backward()
g_att = x.grad[0].norm(dim=-1).clone()

plt.figure(figsize=(7, 3.2))
plt.semilogy(range(1, T + 1), g_rnn.numpy(), label="RNN (tanh)")
plt.semilogy(range(1, T + 1), g_att.numpy(), label="autoatenção")
plt.xlabel("posição do token de entrada (a saída medida é a da posição 50)")
plt.ylabel("‖∂saída₅₀ / ∂xₜ‖")
plt.title("Quanto a saída final depende de cada entrada")
plt.legend(); plt.tight_layout(); plt.show()
print(f"RNN: gradiente da posição 1 / posição 49 = {g_rnn[0].item() / g_rnn[-2].item():.2e}")

**Leitura do gráfico:** na RNN o gradiente cai de forma aproximadamente exponencial com a distância; na atenção ele não decai com a distância, porque todos os tokens entram na mesma soma ponderada.

Comparação do paper (Tabela 1), com $n$ = tamanho da sequência e $d$ = dimensão do modelo:

| Camada | Complexidade por camada | Operações sequenciais | Caminho máximo |
|---|---|---|---|
| Autoatenção | $O(n^2 \cdot d)$ | $O(1)$ | $O(1)$ |
| Recorrente | $O(n \cdot d^2)$ | $O(n)$ | $O(n)$ |

O custo da atenção é quadrático em $n$, e é por isso que modelos como o Whisper trabalham com janelas fixas (30 s de áudio).

## 2. Embeddings e codificação posicional

**Embedding.** Cada token (um inteiro) vira um vetor de dimensão $d_{model}$ por uma tabela aprendida, `nn.Embedding(vocab, d_model)`. É uma consulta de linha: não há conta, só indexação.

In [ ]:
emb = nn.Embedding(num_embeddings=13, embedding_dim=8)
tokens = torch.tensor([[3, 4, 5, 3]])          # batch=1, T=4 (note: o token 3 aparece duas vezes)
e = emb(tokens)
print("tokens:", tuple(tokens.shape), "-> embeddings:", tuple(e.shape))
print("mesmo token, mesmo vetor (posição 0 e 3)?", torch.equal(e[0, 0], e[0, 3]))

**Por que precisamos de posição?** A atenção trata a entrada como um **conjunto**: se embaralharmos os tokens, a saída é a mesma, só que embaralhada igual (equivariância a permutações). Sem informação extra, "o gato mordeu o cão" e "o cão mordeu o gato" são indistinguíveis. Vamos verificar:

In [ ]:
torch.manual_seed(1)
x = torch.randn(1, 5, 8)
perm = torch.tensor([3, 0, 4, 1, 2])

y = self_attention_demo(x)
y_perm = self_attention_demo(x[:, perm])
print("atenção(x embaralhado) == atenção(x) embaralhado?", torch.allclose(y_perm, y[:, perm], atol=1e-6))

**Solução do paper:** somar ao embedding um vetor que depende só da posição, formado por senos e cossenos de frequências diferentes:

$$PE_{(pos,\,2i)}=\sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right),\qquad PE_{(pos,\,2i+1)}=\cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Dimensões baixas oscilam rápido (distinguem posições vizinhas); dimensões altas oscilam devagar (distinguem regiões distantes), como os dígitos de um relógio ou de um contador binário. Uma propriedade citada no paper: $PE_{pos+k}$ é uma função linear de $PE_{pos}$, o que facilita aprender atenção por deslocamento relativo.

In [ ]:
class PositionalEncoding(nn.Module):
    """Codificação posicional senoidal (Vaswani et al., 2017, seção 3.5).

    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """

    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)                                        # (max_len, 1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))   # (d_model/2,)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))                                     # (1, max_len, d_model)

    def forward(self, x):                                                               # x: (B, T, d_model)
        return self.dropout(x + self.pe[:, : x.size(1)])

In [ ]:
pe = PositionalEncoding(d_model=64, max_len=100, dropout=0.0).pe[0]     # (100, 64)
print("PE:", tuple(pe.shape))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
im = ax[0].imshow(pe[:50].numpy(), aspect="auto", cmap="RdBu")
ax[0].set_xlabel("dimensão"); ax[0].set_ylabel("posição"); ax[0].set_title("Codificação posicional (50 posições)")
plt.colorbar(im, ax=ax[0])

for p in (0, 20, 40):
    sims = (pe[p] @ pe[p : p + 30].T) / 32      # dividido por d_model/2 só para escala
    ax[1].plot(range(30), sims.numpy(), label=f"pos = {p}")
ax[1].set_xlabel("deslocamento k"); ax[1].set_ylabel("PE[pos] · PE[pos+k] / 32")
ax[1].set_title("A similaridade depende (quase só) da distância"); ax[1].legend()
plt.tight_layout(); plt.show()

Note no gráfico da direita que as três curvas coincidem: o produto escalar entre duas posições depende apenas da distância $k$ entre elas. (Exatamente: $PE_{pos}\cdot PE_{pos+k}=\sum_i \cos(k\,\omega_i)$, que não depende de $pos$.)

> **Whisper (aula 2):** o *encoder* usa exatamente essa codificação senoidal, enquanto o *decoder* usa embeddings posicionais **aprendidos**.

## 3. Atenção por produto escalar (*scaled dot-product attention*)

Cada token gera três vetores:

- **Query (Q):** "o que estou procurando?"
- **Key (K):** "o que eu ofereço para ser encontrado?"
- **Value (V):** "o que eu entrego se me escolherem?"

$$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Passos: (1) $QK^T$ mede a compatibilidade entre cada query e cada key; (2) dividimos por $\sqrt{d_k}$; (3) o `softmax` transforma cada linha em pesos que somam 1; (4) a saída é a média ponderada dos *values*.

In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    """Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V

    q: (..., Tq, d_k)   k: (..., Tk, d_k)   v: (..., Tk, d_v)
    mask: booleano, True = pode olhar, False = bloqueado; broadcast para (..., Tq, Tk)
    """
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)                 # (..., Tq, Tk)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)                               # cada linha soma 1
    return weights @ v, weights

### Exemplo numérico conferível à mão

3 tokens, $d_k = 2$. Para a **query 0** $=[1,0]$ contra as keys $[1,0]$, $[0,1]$ e $[1,1]$:

- produtos escalares: $[1,\;0,\;1]$; divididos por $\sqrt 2$: $[0{,}7071,\;0,\;0{,}7071]$
- $e^{0{,}7071}=2{,}0281$ e $e^0=1$, soma $=5{,}0562$ ⇒ pesos $=[0{,}4011,\;0{,}1978,\;0{,}4011]$
- saída $=0{,}4011\cdot v_0+0{,}1978\cdot v_1+0{,}4011\cdot v_2$

In [ ]:
Q = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
K = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
V = torch.tensor([[1., 2.], [3., 4.], [5., 6.]])

out, w = scaled_dot_product_attention(Q, K, V)
print("pesos de atenção (cada linha soma 1):\n", w)
print("\nsaída:\n", out)
print("\nlinha 0 à mão:", (0.4011 * V[0] + 0.1978 * V[1] + 0.4011 * V[2]).tolist())

### Máscaras

Passando `mask`, as posições bloqueadas recebem $-\infty$ antes do softmax e, portanto, peso exatamente 0. Dois usos:

- **Máscara de padding:** ignora posições de preenchimento.
- **Máscara causal** (decoder): a posição $i$ só enxerga $j \le i$. Sem ela, o decoder "colaria" olhando a resposta no treino.

In [ ]:
def make_pad_mask(seq):
    """(B, T) -> (B, 1, 1, T); True onde NÃO é padding."""
    return (seq != PAD).unsqueeze(1).unsqueeze(2)


def make_causal_mask(T, device=None):
    """(1, 1, T, T); True na diagonal e abaixo dela (cada posição só vê o passado e ela mesma)."""
    return torch.tril(torch.ones(T, T, dtype=torch.bool, device=device)).unsqueeze(0).unsqueeze(0)

mask = make_causal_mask(3)[0, 0]
print("máscara causal (True = pode olhar):\n", mask.int())
_, w = scaled_dot_product_attention(Q, K, V, mask)
print("\npesos com máscara causal (triangular inferior):\n", w)

### Por que dividir por $\sqrt{d_k}$?

Se as componentes de $q$ e $k$ têm variância 1, o produto escalar tem variância $d_k$. Para $d_k$ grande, os scores ficam enormes, o softmax **satura** (um peso ≈ 1, os demais ≈ 0) e o gradiente quase some. Dividir por $\sqrt{d_k}$ devolve variância 1.

In [ ]:
torch.manual_seed(0)
dk = 512
q = torch.randn(1000, dk); k = torch.randn(1000, dk)
raw = (q * k).sum(-1)
print(f"desvio padrão do score sem escala: {raw.std():.1f} (esperado ≈ √{dk} = {math.sqrt(dk):.1f})")
print(f"desvio padrão com escala         : {(raw / math.sqrt(dk)).std():.2f}")

scores = torch.randn(10) * math.sqrt(dk)        # 10 chaves, scores "grandes"
print("\nmaior peso do softmax SEM escala:", F.softmax(scores, -1).max().item())
print("maior peso do softmax COM escala:", F.softmax(scores / math.sqrt(dk), -1).max().item())

## 4. Atenção multi-cabeça

Em vez de uma atenção com vetores de tamanho $d_{model}$, usamos $h$ atenções em paralelo, cada uma em um subespaço de tamanho $d_k = d_{model}/h$. Cada cabeça pode especializar-se (posição vizinha, mesmo tipo de token, alinhamento com outra sequência...).

$$\text{MultiHead}(Q,K,V)=\text{Concat}(\text{head}_1,\dots,\text{head}_h)\,W^O,\qquad \text{head}_i=\text{Attention}(QW_i^Q,\,KW_i^K,\,VW_i^V)$$

**Implementação:** em vez de $h$ matrizes separadas, usamos uma única `nn.Linear(d_model, d_model)` para Q, K e V, e depois fatiamos com `view` + `transpose`:

`(B, T, D)` → `view` → `(B, T, h, d_k)` → `transpose(1, 2)` → `(B, h, T, d_k)`

Assim a atenção roda em todas as cabeças de uma vez (a dimensão `h` vira dimensão de lote). No final, `transpose` e `view` desfazem a divisão e a projeção $W^O$ mistura as cabeças.

Note que `MultiHeadAttention.forward(x_q, x_kv)` recebe duas entradas: na **autoatenção** elas são iguais; na **atenção cruzada** (*cross-attention*) $Q$ vem do decoder e $K, V$ vêm do encoder.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Atenção multi-cabeça (seção 3.2.2): h cabeças em paralelo, cada uma com d_k = d_model / h."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model precisa ser divisível por n_heads"
        self.h = n_heads
        self.d_k = d_model // n_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.last_attn = None                                         # guardamos os pesos para visualizar

    def split_heads(self, x):                                         # (B, T, D) -> (B, h, T, d_k)
        B, T, _ = x.shape
        return x.view(B, T, self.h, self.d_k).transpose(1, 2)

    def forward(self, x_q, x_kv, mask=None):
        q = self.split_heads(self.w_q(x_q))                           # Q vem de x_q
        k = self.split_heads(self.w_k(x_kv))                          # K e V vêm de x_kv
        v = self.split_heads(self.w_v(x_kv))                          # (self-attention: x_kv = x_q)
        out, weights = scaled_dot_product_attention(q, k, v, mask)    # (B, h, Tq, d_k)
        self.last_attn = weights.detach()
        B, _, Tq, _ = out.shape
        out = out.transpose(1, 2).contiguous().view(B, Tq, self.h * self.d_k)   # concatena as cabeças
        return self.w_o(out)

In [ ]:
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=64, n_heads=4)
x = torch.randn(2, 10, 64)                       # (B=2, T=10, d_model=64)
y = mha(x, x)
print("entrada:", tuple(x.shape), "-> saída:", tuple(y.shape))
print("pesos de atenção:", tuple(mha.last_attn.shape), " = (B, cabeças, Tq, Tk)")
print("cada linha soma 1?", torch.allclose(mha.last_attn.sum(-1), torch.ones(2, 4, 10)))

# Verificação contra a implementação oficial do PyTorch (mesmos Q, K, V)
q, k, v = (mha.split_heads(l(x)) for l in (mha.w_q, mha.w_k, mha.w_v))
ref = F.scaled_dot_product_attention(q, k, v)
mine, _ = scaled_dot_product_attention(q, k, v)
print("igual a F.scaled_dot_product_attention?", torch.allclose(ref, mine, atol=1e-5))

## 5. Blocos do encoder e do decoder

Cada bloco empilha **subcamadas**, e cada subcamada é envolvida por uma conexão residual e uma normalização:

$$\text{saída}=\text{LayerNorm}\big(x+\text{Dropout}(\text{Sublayer}(x))\big)$$

- **Residual** (`x + ...`): cria um atalho para o gradiente e permite empilhar muitas camadas.
- **LayerNorm:** normaliza cada token (média 0, variância 1 sobre `d_model`) e estabiliza o treino. O paper usa *pós-norm*, como aqui; o Whisper usa *pré-norm* (normaliza **antes** da subcamada).
- **FFN** (*feed-forward*): $\text{FFN}(x)=\max(0,xW_1+b_1)W_2+b_2$, aplicada **a cada posição separadamente**. É onde ocorre boa parte da "memória" do modelo. No paper, $d_{ff}=4\,d_{model}$.

| Bloco | Subcamadas |
|---|---|
| **Encoder** | autoatenção (sem máscara causal) → FFN |
| **Decoder** | autoatenção **mascarada** → atenção cruzada (Q do decoder, K/V do encoder) → FFN |

In [ ]:
class EncoderBlock(nn.Module):
    """Self-attention + FFN, cada subcamada com residual e LayerNorm (pós-norm, como no paper)."""

    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        x = self.norm1(x + self.drop(self.attn(x, x, src_mask)))      # LayerNorm(x + Sublayer(x))
        x = self.norm2(x + self.drop(self.ffn(x)))
        return x


class DecoderBlock(nn.Module):
    """Masked self-attention + cross-attention (olha para o encoder) + FFN."""

    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, y, memory, src_mask, tgt_mask):
        y = self.norm1(y + self.drop(self.self_attn(y, y, tgt_mask)))            # Q, K, V = decoder (com máscara causal)
        y = self.norm2(y + self.drop(self.cross_attn(y, memory, src_mask)))      # Q = decoder; K, V = saída do encoder
        y = self.norm3(y + self.drop(self.ffn(y)))
        return y

In [ ]:
enc = EncoderBlock(64, 4, 128, 0.1).eval()
dec = DecoderBlock(64, 4, 128, 0.1).eval()
x = torch.randn(2, 7, 64)          # lado do encoder: Ts = 7
y = torch.randn(2, 5, 64)          # lado do decoder: Tt = 5

print("EncoderBlock:", tuple(x.shape), "->", tuple(enc(x, None).shape))
print("DecoderBlock:", tuple(y.shape), "->", tuple(dec(y, x, None, make_causal_mask(5)).shape))
print("cross-attention (Tt x Ts):", tuple(dec.cross_attn.last_attn.shape), "= (B, h, Tt, Ts)")

## 6. O Transformer completo

```
src ──► Embedding×√d ──► + PE ──► [EncoderBlock × N] ──► memory ─────────────┐
                                                                               ▼ (K, V da cross-attention)
tgt ──► Embedding×√d ──► + PE ──► [DecoderBlock × N] ──► Linear ──► logits (B, Tt, vocab)
```

- O embedding é multiplicado por $\sqrt{d_{model}}$ para ficar na mesma escala da codificação posicional.
- No **treino**, o decoder recebe a resposta deslocada (*teacher forcing*) e prevê todos os tokens em paralelo, graças à máscara causal.
- Na **inferência**, a saída é gerada **token a token** (autorregressiva): o encoder roda uma vez e o decoder roda a cada passo.

In [ ]:
# Vocabulário da tarefa de brinquedo: inverter sequências de dígitos
PAD, BOS, EOS = 0, 1, 2          # tokens especiais
DIGIT0 = 3                       # o dígito d vira o token DIGIT0 + d
VOCAB = DIGIT0 + 10              # 13 tokens no total

class Transformer(nn.Module):
    def __init__(self, vocab=VOCAB, d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.1, max_len=64):
        super().__init__()
        self.d_model = d_model
        self.src_emb = nn.Embedding(vocab, d_model, padding_idx=PAD)
        self.tgt_emb = nn.Embedding(vocab, d_model, padding_idx=PAD)
        self.pos = PositionalEncoding(d_model, max_len, dropout)
        self.encoder = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.decoder = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.out = nn.Linear(d_model, vocab)                          # projeção final para o vocabulário
        for p in self.parameters():                                   # inicialização Xavier, como no paper
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src, src_mask):
        x = self.pos(self.src_emb(src) * math.sqrt(self.d_model))     # embedding * sqrt(d_model) + PE
        for block in self.encoder:
            x = block(x, src_mask)
        return x                                                      # "memory": (B, Ts, d_model)

    def decode(self, tgt, memory, src_mask, tgt_mask):
        y = self.pos(self.tgt_emb(tgt) * math.sqrt(self.d_model))
        for block in self.decoder:
            y = block(y, memory, src_mask, tgt_mask)
        return y                                                      # (B, Tt, d_model)

    def forward(self, src, tgt):
        src_mask = make_pad_mask(src)                                                    # (B, 1, 1, Ts)
        tgt_mask = make_pad_mask(tgt) & make_causal_mask(tgt.size(1), tgt.device)        # (B, 1, Tt, Tt)
        memory = self.encode(src, src_mask)
        return self.out(self.decode(tgt, memory, src_mask, tgt_mask))                    # logits: (B, Tt, vocab)

In [ ]:
CONFIG = dict(vocab=VOCAB, d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.1, max_len=64)
model = Transformer(**CONFIG)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parâmetros totais: {n_params:,}")
for name, p in model.named_parameters():
    if name.startswith(("src_emb", "encoder.0")):
        print(f"  {name:38s} {tuple(p.shape)}")

**Rastreio de *shapes*:** com *forward hooks* imprimimos o formato da saída de cada módulo principal, para um lote de 2 exemplos (origem de 6 tokens, destino de 5).

In [ ]:
shapes = []
def hook(name):
    def fn(module, inputs, output):
        shapes.append((name, tuple(output.shape)))
    return fn

for name in ["src_emb", "pos", "encoder.0", "encoder.1", "tgt_emb", "decoder.0", "decoder.1", "out"]:
    model.get_submodule(name).register_forward_hook(hook(name))

src = torch.randint(DIGIT0, VOCAB, (2, 6))
tgt = torch.randint(DIGIT0, VOCAB, (2, 5))
model.eval()
logits = model(src, tgt)
for name, shp in shapes:
    print(f"{name:10s} -> {shp}")

## 7. Carregar os pesos e fazer inferência

**Tarefa de brinquedo:** inverter uma sequência de dígitos (`3 1 4 1 5` → `5 1 4 1 3`). É simples, mas obriga o modelo a usar a atenção cruzada para olhar a posição certa da origem, o que facilita visualizar o mecanismo.

**Formato dos dados** (com padding à direita):

| | exemplo |
|---|---|
| `src` | `d1 d2 d3 PAD PAD` |
| `tgt_in` (entrada do decoder) | `BOS d3 d2 d1 PAD PAD` |
| `tgt_out` (alvo) | `d3 d2 d1 EOS PAD PAD` |

In [ ]:
def make_batch(batch_size, min_len=3, max_len=8):
    """Gera pares (src, tgt_in, tgt_out) para a tarefa 'inverter a sequência'.

    src     : d1 d2 d3 PAD PAD            (dígitos, com padding à direita)
    tgt_in  : BOS d3 d2 d1 PAD PAD        (entrada do decoder: deslocada uma posição)
    tgt_out : d3 d2 d1 EOS PAD PAD        (o que o decoder deve prever)
    """
    src = torch.full((batch_size, max_len), PAD, dtype=torch.long)
    tgt_in = torch.full((batch_size, max_len + 1), PAD, dtype=torch.long)
    tgt_out = torch.full((batch_size, max_len + 1), PAD, dtype=torch.long)
    for i in range(batch_size):
        n = random.randint(min_len, max_len)
        digits = torch.randint(0, 10, (n,)) + DIGIT0
        src[i, :n] = digits
        rev = digits.flip(0)
        tgt_in[i, 0] = BOS
        tgt_in[i, 1 : n + 1] = rev
        tgt_out[i, :n] = rev
        tgt_out[i, n] = EOS
    return src, tgt_in, tgt_out


def to_tokens(digits):
    return [DIGIT0 + int(d) for d in digits]


def to_text(tokens):
    """Converte tokens em string de dígitos, parando no EOS."""
    out = []
    for t in tokens:
        t = int(t)
        if t == EOS:
            break
        if t >= DIGIT0:
            out.append(str(t - DIGIT0))
    return "".join(out)

@torch.no_grad()
def greedy_decode(model, src, max_new_tokens=16):
    """Decodificação gulosa autorregressiva: a cada passo, escolhe o token mais provável."""
    model.eval()
    src_mask = make_pad_mask(src)
    memory = model.encode(src, src_mask)                              # o encoder roda UMA vez
    ys = torch.full((src.size(0), 1), BOS, dtype=torch.long, device=src.device)
    finished = torch.zeros(src.size(0), dtype=torch.bool, device=src.device)
    for _ in range(max_new_tokens):
        tgt_mask = make_causal_mask(ys.size(1), src.device)
        h = model.decode(ys, memory, src_mask, tgt_mask)              # decoder roda a cada passo
        next_tok = model.out(h[:, -1]).argmax(-1, keepdim=True)       # só a última posição importa
        ys = torch.cat([ys, next_tok], dim=1)
        finished |= next_tok.squeeze(1) == EOS
        if finished.all():
            break
    return ys[:, 1:]                                                  # remove o BOS


def exact_match_accuracy(model, n=500, min_len=3, max_len=8):
    """Fração de sequências em que a saída gerada é exatamente a sequência invertida."""
    src, _, tgt_out = make_batch(n, min_len, max_len)
    pred = greedy_decode(model, src, max_new_tokens=max_len + 2)
    ok = 0
    for p, t in zip(pred, tgt_out):
        ok += int(to_text(p) == to_text(t))
    return ok / n

In [ ]:
for demo in [make_batch(2)]:
    src_d, tin_d, tout_d = demo
    print("src    :", src_d[0].tolist())
    print("tgt_in :", tin_d[0].tolist())
    print("tgt_out:", tout_d[0].tolist())

Os pesos foram treinados antes da aula por `train_mini_transformer.py`. Aqui só **carregamos** e fazemos inferência.

> Se o arquivo `mini_transformer.pt` não estiver na pasta, rode no terminal: `python train_mini_transformer.py` (cerca de 7 min em uma CPU de 2 núcleos; o acerto já chega a ~100% por volta de 1000–1500 passos, então `--steps 2000` basta).

In [ ]:
def save_checkpoint(model, path="mini_transformer.pt"):
    torch.save({"config": CONFIG, "state_dict": model.state_dict()}, path)


def load_checkpoint(path="mini_transformer.pt"):
    ckpt = torch.load(path, map_location="cpu")
    model = Transformer(**ckpt["config"])
    model.load_state_dict(ckpt["state_dict"])
    return model.eval()

WEIGHTS = "mini_transformer.pt"
assert os.path.exists(WEIGHTS), "Arquivo mini_transformer.pt não encontrado: rode `python train_mini_transformer.py` antes."
model = load_checkpoint(WEIGHTS)
print("Pesos carregados. Parâmetros:", f"{sum(p.numel() for p in model.parameters()):,}")

In [ ]:
random.seed(123)
exemplos = [[3, 1, 4, 1, 5], [9, 8, 7], [2, 7, 1, 8, 2, 8, 1, 8], [0, 0, 1, 0]]

for digitos in exemplos:
    src = torch.full((1, 8), PAD)
    src[0, : len(digitos)] = torch.tensor(to_tokens(digitos))
    pred = greedy_decode(model, src, max_new_tokens=10)
    esperado = "".join(map(str, digitos[::-1]))
    obtido = to_text(pred[0])
    print(f"entrada {''.join(map(str, digitos)):>9s} -> saída {obtido:>9s}   esperado {esperado:>9s}   {'OK' if obtido == esperado else 'ERRO'}")

In [ ]:
random.seed(7)
acc = exact_match_accuracy(model, n=500, min_len=3, max_len=8)
print(f"Acerto exato em 500 sequências novas (comprimentos 3 a 8): {acc:.1%}")

**Observação sobre extrapolação.** O modelo foi treinado com comprimentos de 3 a 8. Vale testar comprimentos maiores (célula abaixo) e discutir: a codificação senoidal não garante, na prática, que o modelo generalize para posições nunca vistas. Isso reaparece como limite de contexto em modelos reais (o Whisper, por exemplo, tem janela fixa de 30 s).

In [ ]:
random.seed(8)
for lo, hi in [(3, 8), (9, 10), (11, 12)]:
    src, _, tgt_out = make_batch(300, lo, hi)
    pred = greedy_decode(model, src, max_new_tokens=hi + 2)
    acc = sum(to_text(p) == to_text(t) for p, t in zip(pred, tgt_out)) / 300
    print(f"comprimentos {lo:2d}-{hi:2d}: acerto exato {acc:.1%}")

## 8. Mapas de atenção: o que o modelo aprendeu?

Como `MultiHeadAttention` guarda `last_attn`, podemos olhar os pesos de cada cabeça. Esperamos ver:

- **Autoatenção do encoder:** padrão livre (todas as posições se enxergam).
- **Autoatenção mascarada do decoder:** triangular inferior (efeito da máscara causal).
- **Atenção cruzada:** para inverter, o token $j$ da saída deve olhar para o token $n-1-j$ da entrada, uma **anti-diagonal**.

*Como ler os gráficos do decoder:* cada linha é o token de **entrada** do decoder naquela posição (`BOS, 9, 5, ...`); o token **previsto** a partir dela é o da linha seguinte (`BOS` prevê `9`, a linha `9` prevê `5`, e assim por diante).

In [ ]:
digitos = [3, 1, 4, 1, 5, 9]
n = len(digitos)
src = torch.tensor([to_tokens(digitos)])
tgt_in = torch.tensor([[BOS] + to_tokens(digitos[::-1])])

model.eval()
with torch.no_grad():
    model(src, tgt_in)                                   # teacher forcing; preenche last_attn

src_lab = [str(d) for d in digitos]
tgt_lab = ["BOS"] + [str(d) for d in digitos[::-1]]

def plot_heads(attn, rows, cols, title):
    # attn: (1, h, Tq, Tk)
    h = attn.size(1)
    fig, axes = plt.subplots(1, h, figsize=(3.0 * h, 3.3))
    for i in range(h):
        axes[i].imshow(attn[0, i].numpy(), cmap="viridis", vmin=0, vmax=1)
        axes[i].set_xticks(range(len(cols))); axes[i].set_xticklabels(cols, fontsize=8)
        axes[i].set_yticks(range(len(rows))); axes[i].set_yticklabels(rows, fontsize=8)
        axes[i].set_title(f"cabeça {i}", fontsize=9)
    fig.suptitle(title, y=1.02); plt.tight_layout(); plt.show()

plot_heads(model.encoder[1].attn.last_attn, src_lab, src_lab, "Encoder · camada 2 · autoatenção (linhas = query, colunas = key)")
plot_heads(model.decoder[1].self_attn.last_attn, tgt_lab, tgt_lab, "Decoder · camada 2 · autoatenção mascarada")
plot_heads(model.decoder[1].cross_attn.last_attn, tgt_lab, src_lab, "Decoder · camada 2 · atenção cruzada (linhas = saída, colunas = entrada)")

**Para discutir em sala**

1. A cross-attention forma a anti-diagonal em todas as cabeças? E na camada 1 (`model.decoder[0].cross_attn`)?
2. O que a máscara causal faz com o gráfico da autoatenção do decoder?
3. Quantos parâmetros tem cada parte (embeddings, atenção, FFN, saída)? Dica: `named_parameters()`.

## Exercícios para casa

1. Troque a tarefa para **soma de dois números** ou **ordenação** alterando apenas `make_batch`, retreine e compare a convergência.
2. Varie `n_heads` (1, 2, 4, 8) e `n_layers` (1, 2, 4): o que muda no acerto e nos mapas de atenção?
3. Troque a normalização para **pré-norm** (`x + sub(norm(x))`), como no Whisper, e compare.
4. Treine com sequências de até 10 dígitos e repita o teste de extrapolação.

---
**Próxima aula (notebook 2):** o mesmo encoder-decoder, agora com entrada de **áudio**: o Whisper, camada a camada.

## Anexo — código de treino (não executar em aula)

Esta é a função usada em `train_mini_transformer.py`. Ela só é definida aqui; para treinar, rode o script no terminal, ou execute `train(Transformer(**CONFIG))` em uma célula.

In [ ]:
def train(model, steps=6000, batch_size=128, lr=1e-3, warmup=300, log_every=500):
    """Loop de treino com teacher forcing. Em aula, NÃO rode: carregue os pesos prontos."""
    opt = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
    t0 = time.time()
    for step in range(1, steps + 1):
        model.train()
        for g in opt.param_groups:                                    # warmup linear, depois decaimento linear
            g["lr"] = lr * min(step / warmup, max(0.05, 1 - (step - warmup) / (steps - warmup + 1)))
        src, tgt_in, tgt_out = make_batch(batch_size)
        logits = model(src, tgt_in)                                   # (B, Tt, vocab)
        loss = loss_fn(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if step % log_every == 0 or step == steps:
            acc = exact_match_accuracy(model, n=300)
            print(f"passo {step:5d} | loss {loss.item():.4f} | acerto exato {acc:.1%} | {time.time() - t0:.0f}s")
    return model